# Model 3 — MSTL + ETS

Uses `timeSeries/dataset/train.csv`, `val.csv`, and `test.csv`; target `trip_count`, modeled per `PULocationID`. No baseline is included. Select ETS settings by validation MAE, refit on train + validation, and evaluate on untouched test. Metrics: MAE, RMSE, R², MAPE (excluding and reporting zero actuals). Forecasts are fixed-origin; no actual future demand is fed into predictions. Missing zone-hours are filled with zero; verify this is appropriate for your data.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.seasonal import MSTL
from statsmodels.tsa.holtwinters import ExponentialSmoothing
warnings.filterwarnings("ignore")

cwd = Path.cwd()
ROOT = cwd if (cwd / "timeSeries" / "dataset").exists() else (cwd.parent if cwd.name == "timeSeries" else cwd)
DATA = ROOT / "timeSeries" / "dataset"
OUT = ROOT / "timeSeries" / "results" / "model_3"
OUT.mkdir(parents=True, exist_ok=True)
files = {n: DATA / f"{n}.csv" for n in ("train", "val", "test")}
for p in files.values():
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}; open notebook from repository root or update DATA.")

dfs = {n: pd.read_csv(p, parse_dates=["pickup_hour"]) for n, p in files.items()}
required = {"PULocationID", "pickup_hour", "trip_count"}
for n, df in dfs.items():
    missing = required - set(df.columns)
    if missing: raise ValueError(f"{n}.csv missing {sorted(missing)}")
    df["trip_count"] = pd.to_numeric(df["trip_count"], errors="coerce")
    df.dropna(subset=["PULocationID", "pickup_hour", "trip_count"], inplace=True)
    df.sort_values(["PULocationID", "pickup_hour"], inplace=True)
    print(f"{n}: rows={len(df):,}; zones={df.PULocationID.nunique()}; "
          f"range={df.pickup_hour.min()} -> {df.pickup_hour.max()}")
train, val, test = dfs["train"], dfs["val"], dfs["test"]
assert train.pickup_hour.max() < val.pickup_hour.min() < test.pickup_hour.min()
print("Chronological split checks passed.")

train: rows=403,374; zones=138; range=2025-09-08 00:00:00 -> 2026-01-07 18:00:00
val: rows=86,388; zones=138; range=2026-01-07 19:00:00 -> 2026-02-02 20:00:00
test: rows=86,526; zones=138; range=2026-02-02 21:00:00 -> 2026-02-28 23:00:00
Chronological split checks passed.


In [2]:
ETS_GRID = [
    {"trend": None, "damped_trend": False},
    {"trend": "add", "damped_trend": False},
    {"trend": "add", "damped_trend": True},
]
PERIODS = (24, 168)
MIN_POINTS = 24 * 14

def series_for(frame, zone):
    s = frame.loc[frame.PULocationID == zone].groupby("pickup_hour")["trip_count"].sum().sort_index()
    if s.empty: return pd.Series(dtype=float)
    idx = pd.date_range(s.index.min(), s.index.max(), freq="h")
    s = s.reindex(idx).fillna(0.0).astype(float)
    s.index.name = "pickup_hour"
    return s

def score(actual, pred):
    a, p = np.asarray(actual, float), np.asarray(pred, float)
    ok = np.isfinite(a) & np.isfinite(p)
    a, p = a[ok], p[ok]
    if not len(a):
        return {"n_rows": 0, "MAE": np.nan, "RMSE": np.nan, "R2": np.nan,
                "MAPE_pct": np.nan, "mape_excluded_zero_count": 0, "mape_excluded_zero_pct": np.nan}
    nz = a != 0
    return {"n_rows": len(a), "MAE": mean_absolute_error(a, p),
            "RMSE": float(np.sqrt(mean_squared_error(a, p))),
            "R2": r2_score(a, p) if len(a) > 1 else np.nan,
            "MAPE_pct": float(np.mean(np.abs((a[nz]-p[nz])/a[nz]))*100) if nz.any() else np.nan,
            "mape_excluded_zero_count": int((~nz).sum()),
            "mape_excluded_zero_pct": float((~nz).mean()*100)}

def forecast_mstl_ets(y, future_index, config):
    if len(y) < MIN_POINTS: raise ValueError(f"only {len(y)} hourly points")
    decomposition = MSTL(y, periods=list(PERIODS), stl_kwargs={"robust": True}).fit()
    seasonal = decomposition.seasonal
    if isinstance(seasonal, pd.Series): seasonal = seasonal.to_frame()
    deseasonal = y.to_numpy(float) - seasonal.sum(axis=1).to_numpy(float)
    ets = ExponentialSmoothing(
        deseasonal, trend=config["trend"],
        damped_trend=config["damped_trend"] if config["trend"] else False,
        seasonal=None, initialization_method="estimated"
    ).fit(optimized=True)
    base_forecast = np.asarray(ets.forecast(len(future_index)), float)
    # Repeat each estimated seasonal component at its matching hourly period.
    season_forecast = np.zeros(len(future_index), float)
    for col in seasonal.columns:
        name = str(col)
        period = 24 if "24" in name else 168
        component = np.asarray(seasonal[col], float)
        season_forecast += np.resize(component[-period:], len(future_index))
    return np.maximum(base_forecast + season_forecast, 0.0)

zones = sorted(set(train.PULocationID) | set(val.PULocationID) | set(test.PULocationID))
print("Zones:", len(zones))

Zones: 138


## Validation-based configuration selection

Each zone is modeled separately. Forecast the full validation horizon from the training endpoint; actual validation demand is used only for scoring. Choose the ETS configuration with the lowest pooled validation MAE.

In [3]:
val_index = pd.date_range(val.pickup_hour.min(), val.pickup_hour.max(), freq="h")
tuning_rows, prediction_rows, failure_rows = [], [], []
for i, zone in enumerate(zones, 1):
    y = series_for(train, zone)
    actual = val.loc[val.PULocationID == zone, ["PULocationID", "pickup_hour", "trip_count"]]
    if y.empty or actual.empty: continue
    if len(y) < MIN_POINTS:
        failure_rows.append({"split":"validation","PULocationID":zone,"config":"all","error":"insufficient history"})
        continue
    for cfg in ETS_GRID:
        label = f"MSTL_ETS_trend-{cfg['trend']}_damped-{cfg['damped_trend']}"
        try:
            pred = pd.Series(forecast_mstl_ets(y, val_index, cfg), index=val_index)
            rows = [{"PULocationID":zone,"pickup_hour":r.pickup_hour,"actual":r.trip_count,
                     "predicted":float(pred.loc[r.pickup_hour]),"model":label}
                    for r in actual.itertuples(index=False)]
            prediction_rows.extend(rows)
            tuning_rows.append({"PULocationID":zone,"model":label,**cfg,
                                **score([r["actual"] for r in rows],[r["predicted"] for r in rows])})
        except Exception as e:
            failure_rows.append({"split":"validation","PULocationID":zone,"config":label,
                                 "error":f"{type(e).__name__}: {e}"})
    if i % 10 == 0: print(f"Processed {i}/{len(zones)} zones")

tuning = pd.DataFrame(tuning_rows)
val_candidates = pd.DataFrame(prediction_rows)
if val_candidates.empty: raise RuntimeError("No successful MSTL+ETS fits; inspect failures.")
tuning.to_csv(OUT/"zone_tuning_results.csv", index=False)
val_candidates.to_csv(OUT/"validation_candidate_predictions.csv", index=False)
summary = pd.DataFrame([{"model":name,**score(g.actual,g.predicted)}
                        for name,g in val_candidates.groupby("model")]).sort_values("MAE")
summary.to_csv(OUT/"validation_candidate_summary.csv", index=False)
display(summary)
best_label = summary.iloc[0].model
best_cfg = next(c for c in ETS_GRID if f"MSTL_ETS_trend-{c['trend']}_damped-{c['damped_trend']}" == best_label)
print("Selected by validation MAE:", best_cfg)

Processed 10/138 zones
Processed 20/138 zones
Processed 30/138 zones
Processed 40/138 zones
Processed 50/138 zones
Processed 60/138 zones
Processed 70/138 zones
Processed 80/138 zones
Processed 90/138 zones
Processed 100/138 zones
Processed 110/138 zones
Processed 120/138 zones
Processed 130/138 zones


,model,n_rows,MAE,RMSE,R2,MAPE_pct,mape_excluded_zero_count,mape_excluded_zero_pct
2,MSTL_ETS_trend-add_damped-True,86388,12.857746,31.504011,0.800471,84.626523,23887,27.650831
0,MSTL_ETS_trend-None_damped-False,86388,12.857796,31.504137,0.800470,84.634698,23887,27.650831
1,MSTL_ETS_trend-add_damped-False,86388,13.207267,32.122436,0.792561,83.874918,23887,27.650831


Selected by validation MAE: {'trend': 'add', 'damped_trend': True}


## Final evaluation

Report the selected model on validation, then refit the pipeline using train + validation and evaluate on the untouched test set. No baseline is included.

In [4]:
def evaluate_partition(history_frame, eval_frame, split):
    future_idx = pd.date_range(eval_frame.pickup_hour.min(), eval_frame.pickup_hour.max(), freq="h")
    rows, errs = [], []
    for zone in zones:
        actual = eval_frame.loc[eval_frame.PULocationID == zone,
                                ["PULocationID","pickup_hour","trip_count"]]
        if actual.empty: continue
        y = series_for(history_frame, zone)
        if y.empty: continue
        try:
            pred = pd.Series(forecast_mstl_ets(y, future_idx, best_cfg), index=future_idx)
            for r in actual.itertuples(index=False):
                rows.append({"split":split,"PULocationID":zone,"pickup_hour":r.pickup_hour,
                             "actual":r.trip_count,"predicted":float(pred.loc[r.pickup_hour]),
                             "model":"MSTL_ETS"})
        except Exception as e:
            errs.append({"split":split,"PULocationID":zone,"error":f"{type(e).__name__}: {e}"})
    return pd.DataFrame(rows), errs

val_predictions, val_errs = evaluate_partition(train, val, "validation")
train_val = pd.concat([train,val], ignore_index=True).sort_values(["PULocationID","pickup_hour"])
test_predictions, test_errs = evaluate_partition(train_val, test, "test")
predictions = pd.concat([val_predictions,test_predictions], ignore_index=True)
predictions.to_csv(OUT/"predictions.csv", index=False)
metrics_rows = [{"split":split,"model":"MSTL_ETS",**score(g.actual,g.predicted)}
                for split,g in predictions.groupby("split")]
metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(OUT/"metrics.csv", index=False)
pd.DataFrame(failure_rows + val_errs + test_errs).to_csv(OUT/"fit_failures.csv", index=False)
display(metrics_df)
print("Chosen config:", best_cfg)
print("Saved results under:", OUT)

,split,model,n_rows,MAE,RMSE,R2,MAPE_pct,mape_excluded_zero_count,mape_excluded_zero_pct
0,test,MSTL_ETS,86526,11.350173,29.514126,0.819796,95.161659,24013,27.752352
1,validation,MSTL_ETS,86388,12.857746,31.504011,0.800471,84.626523,23887,27.650831


Chosen config: {'trend': 'add', 'damped_trend': True}
Saved results under: /Users/sadhanatp/Desktop/Amrita /Semester VII/23CSE452/Case Study/Group/nyc-taxi-demand-forecasting/timeSeries/results/model_3


## Outputs

Saved in `timeSeries/results/model_3/`: `metrics.csv`, `predictions.csv`, `zone_tuning_results.csv`, `validation_candidate_summary.csv`, `validation_candidate_predictions.csv`, and `fit_failures.csv`. Review failed zones and the assumption that absent zone-hour rows mean zero demand before reporting results.